# Chapter 9: Storing Data

In [17]:
import csv
import os
from urllib.parse import urlparse
from urllib.request import urlopen, urlretrieve

from dotenv import load_dotenv
from bs4 import BeautifulSoup
import requests
import psycopg

## Downloading a single file

In [2]:
html = urlopen("http://www.pythonscraping.com")
bs = BeautifulSoup(html, "lxml")
image_location = bs.find("img", {"alt": "python-logo"})["src"]
urlretrieve (image_location, "logo.jpg")

('logo.jpg', <http.client.HTTPMessage at 0x719efe6b7130>)

## Downloading all internal files of the page

In [8]:
download_dir = "downloaded"
base_url = "https://pythonscraping.com/"
base_netloc = urlparse(base_url).netloc

def get_abs_url(source):
    if urlparse(base_url).netloc == "":
        return base_url + source
    return source


def get_download_path(file_url):
    parsed = urlparse(file_url)
    netloc = parsed.netloc.strip("/")
    path = parsed.path.strip("/")
    localfile = f"{download_dir}/{netloc}/{path}"

    # Remove the filename from the path in order to
    #  make the directory structure leading up to it
    localpath = "/".join(localfile.split("/")[:-1])
    os.makedirs(localpath, exist_ok=True)
    return localfile

In [10]:
html = urlopen(base_url)
bs = BeautifulSoup(html, "lxml")
download_list = bs.find_all(src=True)

for download in download_list:
    file_url = get_abs_url(download["src"])
    if file_url is None:
        break

    try:
        urlretrieve(file_url, get_download_path(file_url))
        print(file_url)
    except Exception as e:
        print(f"Could not retrieve {file_url} Error: {e}")

https://pythonscraping.com/wp-includes/js/jquery/jquery.min.js?ver=3.7.1
https://pythonscraping.com/wp-includes/js/jquery/jquery-migrate.min.js?ver=3.4.1
https://pythonscraping.com/wp-content/plugins/pagelayer/js/combined.js?ver=1.7.5
https://pythonscraping.com/wp-content/plugins/email-capture-lead-generation//js/eclg-public.js?ver=1.0.1
https://www.googletagmanager.com/gtag/js?id=GT-TNFZZK6
https://pythonscraping.com/wp-content/uploads/2023/04/python-logo-e1681354047443.png
https://pythonscraping.com/wp-content/uploads/2021/08/home1.jpg
https://pythonscraping.com/wp-content/uploads/2021/08/logo01-e1681353135199.png
https://pythonscraping.com/wp-content/plugins/email-capture-lead-generation//images/ajax_loader.gif
https://pythonscraping.com/wp-content/plugins/contact-form-7/includes/swv/js/index.js?ver=5.7.7
https://pythonscraping.com/wp-content/plugins/contact-form-7/includes/js/index.js?ver=5.7.7
https://pythonscraping.com/wp-content/themes/popularfx/js/navigation.js?ver=1.2.0


## Storing Data to CSV

In [16]:
csv_file = open("test.csv", "w+")

try:
    
    writer = csv.writer(csv_file)
    writer.writerow(("number", "number plus 2", "number times 2"))

    for i in range(10):
        writer.writerow((i, i+2, i*2))
    
finally:
    csv_file.close()

## Retrieve an HTML table and write it as a CSV

In [18]:
html = urlopen(
    "https://en.wikipedia.org/wiki/List_of_countries_with_McDonald's_restaurants"
)
bs = BeautifulSoup(html, "lxml")

HTTPError: HTTP Error 403: Forbidden

In [27]:
url = "https://en.wikipedia.org/wiki/List_of_countries_with_McDonald's_restaurants"
headers = {"User-Agent": "Mr.Robot-Linux_Machine"}

resp = requests.get(url, headers=headers)
bs = BeautifulSoup(resp.text, "lxml")

In [30]:
# The main comparison table is currently the first table on the page
table = bs.find("table", {"class": "wikitable"})
rows = table.find_all("tr")
csv_file = open("mcdonalds_countries_wiki.csv", "w+")
writer = csv.writer(csv_file)

try:
    for row in rows:
        csv_row = []
        for cell in row.find_all(["td", "th"]):
            csv_row.append(cell.get_text(strip=True))
        writer.writerow(csv_row)
finally:
    csv_file.close()

# RDBMS: PostgreSQL

In [6]:
psycopg.__version__

'3.3.4'

In [18]:
load_dotenv("./.env")

True

In [47]:
db_config = {
    "dbname": os.getenv("DB_NAME"),
    "user": os.getenv("DB_USER"),
    "password": os.getenv("DB_PASSWORD"),
    "host": os.getenv("DB_HOST"),
    "port": os.getenv("DB_PORT"),
}

with psycopg.connect(**db_config) as conn:

    for record in conn.execute("SELECT * FROM people;"):
        print(record)
    # -----------------------------------------------------
    # cur = conn.execute("SELECT * FROM people;")
    # for record in cur.fetchall():
    #     print(record)
    # -----------------------------------------------------
    # with conn.cursor() as cur:

    #     cur.execute("SELECT * FROM people;")
    #     for record in cur:
    #         print(record)

    #     # for row in cur.fetchall():
    #        # print(row)

(1, 'Sarah', 'Connor', 28, 'Pilot')
(2, 'Alice', 'Walker', 30, 'DevOps')
(3, 'Bob', 'Builder', 40, 'Architect')
(4, 'John', 'Will', 50, 'Pilot')


In [54]:
with psycopg.connect(**db_config) as conn:
    conn.execute("""
    INSERT INTO people (first_name, last_name, age, job)
    VALUES (%(first_name)s, %(last_name)s, %(age)s, %(job)s)
    ;""",
    {
        "first_name": "Jack",
        "last_name": "Smith",
        "age": 19,
        "job": "student",
    })
    conn.commit()

In [55]:
with psycopg.connect(**db_config) as conn:
    for record in conn.execute("SELECT * FROM people;"):
        print(record)

(1, 'Sarah', 'Connor', 28, 'Pilot')
(2, 'Alice', 'Walker', 30, 'DevOps')
(3, 'Bob', 'Builder', 40, 'Architect')
(4, 'John', 'Will', 50, 'Pilot')
(5, 'Jack', 'Smith', 19, 'student')
